## Regression Predictions of Exam Scores

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [21]:
# Example data
np.random.seed(42)
Student = pd.read_csv("student_performance_factors.csv")

X = Student.iloc[:, :-1]
scores = Student.loc[:, "Exam_Score"]

categorical_cols = ["Parental_Involvement", "Access_to_Resources","Extracurricular_Activities","Motivation_Level",
"Internet_Access", "Family_Income" , "Teacher_Quality"  , "School_Type", "Peer_Influence",    
"Learning_Disabilities", "Parental_Education_Level",   "Distance_from_Home" , "Gender"]

numeric = ["Hours_Studied", "Attendance", "Sleep_Hours", "Previous_Scores", "Tutoring_Sessions","Physical_Activity"]


dummies_drop = pd.get_dummies(X[categorical_cols], drop_first=True)

Xdummies = pd.concat([X[numeric], dummies_drop], axis=1)

poly = PolynomialFeatures(degree=2, include_bias=True)

X_interactions = poly.fit_transform(Xdummies)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_interactions, scores, test_size=0.2, random_state=42
)

# Scale features (important for Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create LassoCV model
lasso_cv = LassoCV(
    cv=5,                  # 5-fold cross-validation
    random_state=42,
    alphas=100,           # Auto-generate alpha values
    max_iter=10000
)

# Fit the model
lasso_cv.fit(X_train_scaled, y_train)

# Print results
print(f"Best alpha: {lasso_cv.alpha_:.6f}")
print(f"R² Score: {lasso_cv.score(X_test_scaled, y_test):.4f}")

# Make predictions
y_pred = lasso_cv.predict(X_test_scaled)
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

Best alpha: 0.027009
R² Score: 0.7644
RMSE: 1.8249


In [22]:
np.sum(lasso_cv.coef_ != 0)

np.int64(133)

## Classification using Logistic regression with l1 penalty

In [48]:
X = Student.drop('Family_Income', axis = 1)
types = Student.loc[:,'Family_Income']

categorical_cols = ["Parental_Involvement", "Access_to_Resources","Extracurricular_Activities","Motivation_Level",
"Internet_Access", "School_Type" , "Teacher_Quality", "Peer_Influence",    
"Learning_Disabilities", "Parental_Education_Level",   "Distance_from_Home" , "Gender"]

numeric = ["Hours_Studied", "Attendance", "Sleep_Hours", "Previous_Scores", "Tutoring_Sessions","Physical_Activity", "Exam_Score"]


dummies_drop = pd.get_dummies(X[categorical_cols], drop_first=True)

Xdummies = pd.concat([X[numeric], dummies_drop], axis=1)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    Xdummies, types, test_size=0.2, random_state=42
)

# Scale features (important for Lasso)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [49]:
# Logistic Lasso with built-in cross-validation for C
logistic_lasso_cv = LogisticRegressionCV(
    scoring='balanced_accuracy',
    penalty = 'l1',
    solver = "saga",
    Cs=100,                  # Number of C values to try
    cv=5,                   # 5-fold cross-validation
    max_iter=10000,
    random_state=42
)

# Fit with CV
mod = logistic_lasso_cv.fit(X_train_scaled, y_train)

# Results
print("="*50)
print("Logistic Lasso with Cross-Validation")
print("="*50)
print(f"Best C: {logistic_lasso_cv.C_[0]:.6f}")
print(f"Test Accuracy: {mod.score(X_test_scaled, y_test):.4f}")

/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new be

Logistic Lasso with Cross-Validation
Best C: 0.756463
Test Accuracy: 0.4870


In [46]:
X_train_scaled.shape

(5285, 36)

In [28]:
2672/6607

0.4044195550174058